# Experiment 4: Implementation of Regression Models with Regularization (Ridge, Lasso & ElasticNet)

**Course:** BCSE0731 — Machine Learning Lab  
**Aim:** Compare unregularized **Linear Regression (OLS)** against **Ridge ($L_2$)**, **Lasso ($L_1$)**, and **ElasticNet** to control overfitting, inspect coefficient shrinkage paths, and select optimal regularization strength $\alpha$.

### Cost Functions
- **Ridge ($L_2$):** $\text{MSE} + \alpha \sum w_j^2$ (shrinks weights smoothly, keeps all features)
- **Lasso ($L_1$):** $\text{MSE} + \alpha \sum |w_j|$ (drives weak weights to **exact 0** $\rightarrow$ feature selection)
- **ElasticNet:** $\text{MSE} + \alpha [r \sum |w_j| + \frac{1-r}{2} \sum w_j^2]$

### Step 1: Import Libraries & Load Dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score

### Step 2: Create High-Dimensional Polynomial Features (Overfitting Scenario)

In [ ]:
data = load_diabetes()
X, y = data.data, data.target

# Expand to degree-2 polynomial features (65 features) to induce potential overfitting
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
feature_names = poly.get_feature_names_out(data.feature_names)

X_train, X_test, y_train, y_test = train_test_split(X_poly, y, test_size=0.25, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
print("Expanded Feature Count:", X_train_sc.shape[1])

### Step 3: Compare OLS, Ridge, Lasso & ElasticNet

In [ ]:
models = {
    'OLS LinearRegression': LinearRegression(),
    'Ridge (alpha=50.0)': Ridge(alpha=50.0),
    'Lasso (alpha=2.0)': Lasso(alpha=2.0, max_iter=10000),
    'ElasticNet (alpha=1.0)': ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=10000)
}

rows = []
for name, m in models.items():
    m.fit(X_train_sc, y_train)
    train_r2 = r2_score(y_train, m.predict(X_train_sc))
    test_r2 = r2_score(y_test, m.predict(X_test_sc))
    test_rmse = np.sqrt(mean_squared_error(y_test, m.predict(X_test_sc)))
    nonzero = np.sum(np.abs(m.coef_) > 1e-5)
    rows.append({'Model': name, 'Train R2': round(train_r2, 4), 'Test R2': round(test_r2, 4),
                 'Test RMSE': round(test_rmse, 2), 'Non-Zero Weights': nonzero})

comp_df = pd.DataFrame(rows)
display(comp_df)

### Step 4: Cross-Validated Alpha Tuning & Coefficient Shrinkage Path

In [ ]:
alphas = np.logspace(-2, 3, 40)
ridge_cv = RidgeCV(alphas=alphas).fit(X_train_sc, y_train)
lasso_cv = LassoCV(alphas=np.logspace(-2, 2, 40), max_iter=10000, random_state=42).fit(X_train_sc, y_train)
print(f"Optimal Ridge alpha = {ridge_cv.alpha_:.3f} | Test R2 = {r2_score(y_test, ridge_cv.predict(X_test_sc)):.4f}")
print(f"Optimal Lasso alpha = {lasso_cv.alpha_:.3f} | Test R2 = {r2_score(y_test, lasso_cv.predict(X_test_sc)):.4f}")

# Regularization Path Visualization
ridge_coefs, lasso_coefs = [], []
for a in alphas:
    ridge_coefs.append(Ridge(alpha=a).fit(X_train_sc, y_train).coef_[:10])
    lasso_coefs.append(Lasso(alpha=a, max_iter=10000).fit(X_train_sc, y_train).coef_[:10])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
ax1.plot(alphas, ridge_coefs, lw=1.8)
ax1.set_xscale('log'); ax1.set_title('Ridge (L2): Smooth Coefficient Shrinkage', fontweight='bold')
ax1.set_xlabel('Alpha (log scale)'); ax1.set_ylabel('Coefficient Value'); ax1.grid(True, alpha=0.4)

ax2.plot(alphas, lasso_coefs, lw=1.8)
ax2.set_xscale('log'); ax2.set_title('Lasso (L1): Sparse Zeroing of Weights', fontweight='bold')
ax2.set_xlabel('Alpha (log scale)'); ax2.set_ylabel('Coefficient Value'); ax2.grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

### Conclusion
- Unregularized OLS overfits when polynomial features grow (`Train R2 >> Test R2`).
- **Ridge ($L_2$)** stabilizes weights uniformly while **Lasso ($L_1$)** performs automatic feature selection by zeroing irrelevant coefficients.